In [1]:
# If required, uncomment:
# %pip install pandas numpy scikit-learn matplotlib seaborn joblib

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42
TARGET = "Condition_Class"
CLASS_NAMES = ["Excellent", "Good", "Fair", "Poor"]

In [2]:
# Shared grouped splits + the agreed 30 features (see codd_common.py).
# Never regenerate the split here: all models must use the same files.
from codd_common import load_grouped_splits, assert_feature_set

data = load_grouped_splits()
train_df, val_df, test_df = data.train_df, data.val_df, data.test_df
X_train, X_val, X_test = data.X_train, data.X_val, data.X_test
y_train, y_val, y_test = data.y_train, data.y_val, data.y_test
feature_columns = data.feature_columns
numeric_features = data.numeric_features
categorical_features = data.categorical_features

assert_feature_set(X_train.columns)
assert_feature_set(X_val.columns)
assert_feature_set(X_test.columns)
print("Features used:", feature_columns)

for name, df in [("Training", train_df), ("Validation", val_df), ("Testing", test_df)]:
    print(f"{name}: {df.shape}")
    print(df[TARGET].value_counts().reindex(CLASS_NAMES, fill_value=0))


Grouped splits: train (11465, 30) val (2410, 30) test (2644, 30)
Image_ID overlap (must all be 0): {'train-val': 0, 'train-test': 0, 'val-test': 0}
Features: 30 (24 numeric, 6 categorical)
Features used: ['Material_Type', 'Material_Subtype', 'Source', 'Exposure_Type', 'Crack_Level', 'Surface_Damage', 'Breakage_Level', 'Discoloration_Level', 'Deformation_Level', 'Missing_Portion', 'pixel_obj_width', 'pixel_obj_height', 'relative_area', 'edge_density', 'fill_ratio', 'color_std', 'mean_brightness', 'std_brightness', 'contrast', 'texture_laplacian_var', 'dark_pixel_ratio', 'bright_pixel_ratio', 'mean_hue', 'mean_saturation', 'saturation_std', 'contour_area', 'contour_perimeter', 'circularity', 'solidity', 'extent']
Training: (11465, 43)
Condition_Class
Excellent     128
Good         2460
Fair         7899
Poor          978
Name: count, dtype: int64
Validation: (2410, 43)
Condition_Class
Excellent      21
Good          568
Fair         1596
Poor          225
Name: count, dtype: int64
Testin

In [3]:
print('Numeric:', len(numeric_features), 'Categorical:', len(categorical_features))


Numeric: 24 Categorical: 6


In [4]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
], remainder="drop")

In [5]:
rf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

ada_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=RANDOM_STATE),
        n_estimators=150,
        learning_rate=0.5,
        random_state=RANDOM_STATE
    ))
])

cart_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(
        criterion="gini",
        max_depth=best_cart_depth if "best_cart_depth" in globals() else 12,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

svm_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE
    ))
])

estimators = [
    ("random_forest", rf_pipeline),
    ("ada_boost", ada_pipeline),
    ("cart", cart_pipeline),
    ("svm", svm_pipeline)
]
print("Base estimators defined:", [name for name, _ in estimators])

Base estimators defined: ['random_forest', 'ada_boost', 'cart', 'svm']


In [6]:
hard_voting = VotingClassifier(
    estimators=estimators,
    voting="hard",
    n_jobs=-1
)

soft_voting = VotingClassifier(
    estimators=estimators,
    voting="soft",
    n_jobs=-1,
    weights=None
)

# Fit both ensembles on the same training split.
# AdaBoost's shallow decision-tree estimator uses balanced class weights.
hard_voting.fit(X_train, y_train)
soft_voting.fit(X_train, y_train)

print("Both voting ensembles trained.")

Both voting ensembles trained.


In [7]:
def calculate_metrics(model, X, y, name):
    pred = model.predict(X)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y, pred),
        "Macro Precision": precision_score(y, pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y, pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y, pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y, pred, average="weighted", zero_division=0)
    }

validation_results = pd.DataFrame([
    calculate_metrics(hard_voting, X_val, y_val, "Hard Voting"),
    calculate_metrics(soft_voting, X_val, y_val, "Soft Voting")
]).sort_values("Macro F1", ascending=False)

display(validation_results)

best_voting_name = validation_results.iloc[0]["Model"]
best_voting = hard_voting if best_voting_name == "Hard Voting" else soft_voting
print("Selected ensemble:", best_voting_name)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,Soft Voting,0.99751,0.998493,0.985507,0.991853,0.997505
0,Hard Voting,0.99751,0.998777,0.985224,0.991851,0.997504


Selected ensemble: Soft Voting


In [8]:
# Final ensemble: refit the validation-selected voting type on train + validation
from sklearn.base import clone
X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)
best_voting = clone(best_voting).fit(X_train_val, y_train_val)

y_test_pred = best_voting.predict(X_test)

print("Selected Voting Ensemble:", best_voting_name)
print("Test Accuracy:", round(accuracy_score(y_test, y_test_pred), 4))
print("Macro Precision:", round(precision_score(y_test, y_test_pred, average="macro", zero_division=0), 4))
print("Macro Recall:", round(recall_score(y_test, y_test_pred, average="macro", zero_division=0), 4))
print("Macro F1:", round(f1_score(y_test, y_test_pred, average="macro", zero_division=0), 4))
print("Weighted F1:", round(f1_score(y_test, y_test_pred, average="weighted", zero_division=0), 4))

print("\\nClassification Report:")
print(classification_report(y_test, y_test_pred, labels=CLASS_NAMES, zero_division=0))

Selected Voting Ensemble: Soft Voting
Test Accuracy: 0.9989
Macro Precision: 0.999
Macro Recall: 0.9833
Macro F1: 0.9909
Weighted F1: 0.9989
\nClassification Report:
              precision    recall  f1-score   support

   Excellent       1.00      0.94      0.97        32
        Good       1.00      1.00      1.00       586
        Fair       1.00      1.00      1.00      1788
        Poor       1.00      1.00      1.00       238

    accuracy                           1.00      2644
   macro avg       1.00      0.98      0.99      2644
weighted avg       1.00      1.00      1.00      2644



In [9]:
cm = confusion_matrix(y_test, y_test_pred, labels=CLASS_NAMES)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title(f"{best_voting_name} – Test Confusion Matrix")
plt.tight_layout()
plt.show()

C:\Users\Ryan John\AppData\Local\Temp\ipykernel_27696\312113250.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
os.makedirs("saved_models", exist_ok=True)

joblib.dump(best_voting, "saved_models/codd_voting_ensemble.joblib")
print("Saved selected voting ensemble to saved_models/codd_voting_ensemble.joblib")

# Save the fitted component estimators from the selected VotingClassifier.
for estimator_name, fitted_estimator in zip(
    [name for name, _ in estimators],
    best_voting.estimators_
):
    joblib.dump(
        fitted_estimator,
        f"saved_models/codd_voting_{estimator_name}.joblib"
    )

print("Saved all fitted component estimators.")

Saved selected voting ensemble to saved_models/codd_voting_ensemble.joblib
Saved all fitted component estimators.
